# Example 5 Wedding extended

## Mathematical Model:
$$
\begin{equation*}
	\textrm{max } z =  \sum_{i=1}^{n}\sum_{j=1}^{n}\sum_{t=1}^{T}s_{ij}x_{it}x_{jt}
\end{equation*}
$$

$$
\begin{align*}
	&& \textrm{subject to constraints: }  \sum_{i=1}^{n}x_{it} &\leq K_t && \textrm{for } t = 1,\dots,T \\
	&& \sum_{t=1}^{T}x_{it} &= 1 && \textrm{for }  i = 1,\dots,n \\
	&& x_{it} &\in \{0,1\} &&  \forall i,t \\
\end{align*}
$$


## Problem:
Nine people are to be distributed over three tables so that the sympathy values of the people per table are maximized.

### Sympathy/likeability values

|           | Anna | Berti | Carl | Dieter | Emil | Franz | Gerd | Hanna | Ilse |
|:---------:|:----:|:-----:|:----:|:------:|:----:|:-----:|:----:|:-----:|:----:|
|    Anna   |   0  |   6   |   4  |   10   |   1  |   2   |   6  |   2   |   4  |
|   Berti   |   6  |   0   |   7  |    3   |   6  |   1   |   5  |   10  |   5  |
|    Carl   |   4  |   7   |   0  |   10   |   6  |   3   |  10  |   1   |   4  |
|   Dieter  |  10  |   3   |  10  |    0   |   4  |   6   |   9  |   6   |  10  |
|    Emil   |   1  |   6   |   6  |    4   |   0  |   10  |   1  |   2   |   4  |
|   Franz   |   2  |   1   |   3  |    6   |  10  |   0   |   9  |   7   |   7  |
|    Gerd   |   6  |   5   |  10  |    9   |   1  |   9   |   0  |   1   |   9  |
|   Hanna   |   2  |   10  |   1  |    6   |   2  |   7   |   1  |   0   |   8  |
|    Ilse   |   4  |   5   |   4  |   10   |   4  |   7   |   9  |   8   |   0  |

### Table sizes
The three tables have the following sizes: 4, 3 and 2 seats


In [ ]:
import gurobipy as gp
from gurobipy import GRB
import csv

## Sets - Definition

In [ ]:
numberOfTables = 3
numberOfPersons = 9

tables = range(numberOfTables)
persons = range(numberOfPersons)

## Parameter definition

Sympathy values:

In [ ]:
sympathy=[]
with open("Bsp4_SympathieWerte.csv", encoding="utf-8") as csv_file:
    csv_reader = csv.reader(csv_file)
    names = next(csv_reader)[1:]
    for row in csv_reader:
        rowAsInt = [int(item) for item in row[1:]]
        sympathy.append(rowAsInt)

sympathy

Table sizes:

In [ ]:
capacity = [4, 3, 2]

## Initialising the model

In [ ]:
model = gp.Model()

## Initialising the variables

In [ ]:
x = model.addVars(numberOfPersons, numberOfTables, vtype=GRB.BINARY, name="x")

## Defining the objective function

$$
\begin{equation*}
	\textrm{max } z =  \sum_{i=1}^{n}\sum_{j=1}^{n}\sum_{t=1}^{T}s_{ij}x_{it}x_{jt}
\end{equation*}
$$

In [ ]:
z = gp.quicksum(sympathy[person1][person2] * x[person1, table] * x[person2, table]
                for table in tables for person1 in persons for person2 in persons if person1 != person2)    # if-condition removes unnecessary sum terms

## Adding constraints

$$
\begin{align*}
	&& \textrm{subject to constraints: }  \sum_{i=1}^{n}x_{it} &\leq K_t && \textrm{for } t = 1,\dots,T \\
	&& \sum_{t=1}^{T}x_{it} &= 1 && \textrm{for }  i = 1,\dots,n \\
\end{align*}
$$

In [ ]:
model.addConstrs((x.sum('*',table) <= capacity[table] for table in tables), name="ConstrTable")

model.addConstrs((x.sum(person,'*') == 1 for person in persons), name="ConstrPerson")

## Optimization model

In [ ]:
model.optimize()


## Result output

In [ ]:
if model.Status == GRB.OPTIMAL:
    model.printAttr('ObjVal')
    model.printAttr('X')
elif model.Status == GRB.INFEASIBLE:
    print("Model is not solveable!")

In [ ]:
model.write("model.lp")

In [ ]:
model.write("loesung.sol")

## Intelligent output
Definition of the names:


In [ ]:
# names = ["Anna", "Berti", "Carl", "Dieter", "Emil", "Franz", "Gerd", "Hanna", "Ilse"]

In [ ]:
print("Distribution of seats\n")
for person in persons:
    for table in tables:
        if x[person, table].X > 0:
            print(f"{names[person]}\tsits at the table {table}")

In [ ]:
for table in tables:
    print(f"\nPeople at the table {table}")
    for person1 in range(numberOfPersons-1):
        for person2 in range(person1+1, numberOfPersons):
            if x[person1, table].X * x[person2, table].X > 0:
                print(f"{names[person1]}\tand {names[person2]}\thave the sympathy value: {sympathy[person1][person2]}")